# Vectorless RAG with PageIndex

This notebook builds a small end-to-end demo of **vectorless RAG** using [PageIndex](https://github.com/VectifyAI/PageIndex) (VectifyAI).

**Traditional RAG**: split a document into fixed-size chunks -> embed each chunk -> store vectors in a vector DB -> retrieve by nearest-neighbor similarity search.

**Vectorless RAG (PageIndex)**: parse the document into a hierarchical tree that mirrors its real structure (chapters, sections, subsections) -> ask an LLM to *reason* over the tree (titles + summaries) and decide which node(s) actually answer the question -> read only those sections.

No embeddings. No vector database. No arbitrary chunk boundaries cutting sentences in half. Retrieval is a reasoning/navigation step, not a similarity search -- and it's traceable: you can always point at *which section* the answer came from.

We'll run this **self-hosted / local**: PageIndex builds the tree on your machine via your own `OPENAI_API_KEY` (through LiteLLM under the hood) -- no PageIndex account or API key needed.

## Contents
1. Setup
2. Fetch a sample document
3. Build the PageIndex tree (indexing)
4. Inspect the tree
5. Minimal reasoning-based retrieval (manual tree search)
6. Agentic multi-hop retrieval (`openai-agents`)
7. Vectorless vs. vector RAG -- a side-by-side comparison
8. Cost & caching notes
9. Wrap-up

## 1. Setup

```bash
pip install -r requirements.txt
```

Copy `.env.example` to `.env` and fill in a real `OPENAI_API_KEY` before running the next cell.

In [ ]:
import json
import os
import warnings as wr
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY in a .env file (copy .env.example to .env) before running this notebook."
)
print("OPENAI_API_KEY is set.")

## 2. Fetch a sample document

We use the same structurally-rich arXiv paper PageIndex's own examples use, so the tree-building step is proven to work end-to-end. Swap in any PDF with real section structure (a 10-K, a manual, a long report) by changing `PDF_URL`/`PDF_PATH`.

In [ ]:
import requests

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

PDF_URL = "https://arxiv.org/pdf/2603.15031"
PDF_PATH = DATA_DIR / "sample-paper.pdf"

if not PDF_PATH.exists():
    print(f"Downloading {PDF_URL} ...")
    with requests.get(PDF_URL, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(PDF_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    print("Download complete.")
else:
    print(f"Using cached {PDF_PATH}")

## 3. Build the PageIndex tree (indexing)

`PageIndexLocalClient` runs entirely on your machine -- it uses `OPENAI_API_KEY` (via LiteLLM) to read the PDF and build a hierarchical tree of titles, page ranges, and per-section summaries. We cache the resulting `doc_id` on disk so re-running this notebook doesn't re-index (and re-pay for) the same PDF.

In [ ]:
from pageindex import PageIndexAPIError, PageIndexLocalClient

client = PageIndexLocalClient(storage_path=".pageindex")

DOC_ID_PATH = DATA_DIR / "sample-paper.doc_id"

doc_id = None
if DOC_ID_PATH.exists():
    cached = DOC_ID_PATH.read_text().strip()
    try:
        client.get_document(cached)
        doc_id = cached
    except PageIndexAPIError:
        DOC_ID_PATH.unlink()

if doc_id is None:
    print("Indexing PDF (this calls the LLM to build the tree; can take a minute)...")
    doc_id = client.submit_document(str(PDF_PATH), wait=True)["doc_id"]
    DOC_ID_PATH.write_text(doc_id)

print(f"doc_id: {doc_id}")

## 4. Inspect the tree

This is the payoff: instead of a pile of vectors, you get a readable table of contents an LLM (or you) can navigate.

In [ ]:
from pageindex import utils as pageindex_utils

tree = client.get_tree(doc_id, node_summary=True)["result"]
pageindex_utils.print_tree(tree)

In [ ]:
# Raw shape of a single node, for reference
print(json.dumps(tree[0], indent=2)[:600])

## 5. Minimal reasoning-based retrieval (manual tree search)

The core idea of vectorless RAG in its simplest form: give the LLM the tree's **titles + summaries only** (no full text, no embeddings) and ask it which node(s) are relevant. Then read just those sections and answer. This mirrors PageIndex's own [tree-search tutorial](https://github.com/VectifyAI/PageIndex/blob/main/examples/tutorials/tree-search/README.md).

In [ ]:
from openai import OpenAI

oai = OpenAI()


def flatten_titles_and_summaries(nodes, lines=None):
    if lines is None:
        lines = []
    for n in nodes:
        summary = n.get("summary") or n.get("prefix_summary", "")
        lines.append(f"[{n['node_id']}] {n['title']} -- {summary}")
        if n.get("nodes"):
            flatten_titles_and_summaries(n["nodes"], lines)
    return lines


def find_node(nodes, node_id):
    for n in nodes:
        if n["node_id"] == node_id:
            return n
        if n.get("nodes"):
            found = find_node(n["nodes"], node_id)
            if found:
                return found
    return None


# Structure only -- no node text, so this step never sees (or pays to send) the full document
structure_only = client.get_document_structure(doc_id)
tree_outline = "\n".join(flatten_titles_and_summaries(structure_only))

QUESTION = "What problem does this paper try to solve, and what is its main proposed method?"

search_prompt = f"""You are given a query and the tree structure of a document.
Find all nodes that are likely to contain the answer.

Query: {QUESTION}

Document tree structure:
{tree_outline}

Reply as JSON: {{"thinking": "<your reasoning>", "node_list": ["<node_id>", ...]}}"""

search_response = oai.chat.completions.create(
    model="gpt-4o-2024-11-20",
    messages=[{"role": "user", "content": search_prompt}],
    response_format={"type": "json_object"},
)
search_result = json.loads(search_response.choices[0].message.content)
print(json.dumps(search_result, indent=2))

In [ ]:
# Now read ONLY the selected node(s) -- not the whole document, not a similarity-ranked chunk
full_tree = client.get_tree(doc_id, node_summary=True, include_text=True)["result"]
selected_text = "\n\n".join(
    node["text"] for nid in search_result["node_list"]
    if (node := find_node(full_tree, nid)) and node.get("text")
)

answer_prompt = f"""Answer the question using ONLY the excerpt below.

Question: {QUESTION}

Excerpt:
{selected_text}"""

answer_response = oai.chat.completions.create(
    model="gpt-4o-2024-11-20",
    messages=[{"role": "user", "content": answer_prompt}],
)
print(answer_response.choices[0].message.content)

## 6. Agentic multi-hop retrieval

For questions that need synthesis across multiple sections, PageIndex ships ready-made tool contracts (`browse_documents`, `get_document_structure`, `get_page_content`, ...) plus a retrieval playbook, wired up in one call via `client.openai_agent_config(doc_id=...)`. The agent decides on its own which sections to open, potentially hopping across several, before answering -- and we can watch it do so.

In [ ]:
import asyncio

from agents import Agent, Runner, set_tracing_disabled
from agents.stream_events import RawResponsesStreamEvent, RunItemStreamEvent
from openai.types.responses import ResponseTextDeltaEvent

set_tracing_disabled(True)


async def query_agent(client, doc_id, prompt):
    agent = Agent(**client.openai_agent_config(doc_id=doc_id))
    streamed_run = Runner.run_streamed(agent, prompt)
    async for event in streamed_run.stream_events():
        if isinstance(event, RawResponsesStreamEvent) and isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)
        elif isinstance(event, RunItemStreamEvent) and event.item.type == "tool_call_item":
            raw = event.item.raw_item
            print(f"\n[tool call]: {raw.name}({getattr(raw, 'arguments', '')})")
    print()
    return str(streamed_run.final_output)


MULTI_HOP_QUESTION = "Summarize the proposed method, then explain how the experiments validate it."
final_answer = await query_agent(client, doc_id, MULTI_HOP_QUESTION)

## 7. Vectorless vs. vector RAG

| | Vectorless RAG (PageIndex) | Traditional vector RAG |
|---|---|---|
| Retrieval mechanism | LLM reasons over a structured tree | Nearest-neighbor search over embeddings |
| Infra | None -- no vector DB | Pinecone / Chroma / pgvector, an embedding pipeline |
| Chunking | None -- natural sections | Fixed-size windows that can cut mid-sentence/mid-table |
| Traceability | High -- answer cites a titled section | Low -- a similarity score, not a reason |
| Domain knowledge injection | Add a sentence to the search prompt | Requires re-embedding / fine-tuning |
| Cost profile | LLM call(s) per query for tree search/reasoning | Cheap per-query vector lookup, upfront embedding cost |
| Best fit | Long, well-structured documents (reports, manuals, filings) | Huge corpora where per-query LLM reasoning is too slow/costly |

**Optional stretch cell below**: build a quick naive vector-RAG baseline over the *same* document and *same* question, so you can eyeball how each retrieval strategy picks its context.

In [ ]:
# Optional: naive vector-RAG baseline for a side-by-side comparison.
# Chunks the same tree's text into fixed-size windows, embeds them, and retrieves by cosine similarity.
import numpy as np


def collect_chunks(nodes, chunk_words=400, chunks=None):
    if chunks is None:
        chunks = []
    for n in nodes:
        words = n.get("text", "").split()
        for i in range(0, len(words), chunk_words):
            piece = " ".join(words[i:i + chunk_words])
            if piece.strip():
                chunks.append(piece)
        if n.get("nodes"):
            collect_chunks(n["nodes"], chunk_words, chunks)
    return chunks


def embed(texts):
    resp = oai.embeddings.create(model="text-embedding-3-small", input=texts)
    return np.array([d.embedding for d in resp.data])


chunks = collect_chunks(full_tree)
print(f"{len(chunks)} naive fixed-size chunks (vs. {len(search_result['node_list'])} reasoned-over node(s) above)")

chunk_vecs = embed(chunks)
query_vec = embed([QUESTION])[0]
sims = chunk_vecs @ query_vec / (np.linalg.norm(chunk_vecs, axis=1) * np.linalg.norm(query_vec))
top_idx = int(np.argmax(sims))

print(f"\nTop vector-RAG chunk (cosine similarity={sims[top_idx]:.3f}):\n")
print(chunks[top_idx][:800])

## 8. Cost & caching notes

- **Indexing mode**: `submit_document` defaults to PageIndex's *flash* (heuristic, cheaper) mode; pass `mode="standard"` for an LLM-optimized tree on documents with unclear structure, at higher cost.
- **Caching**: we saved `doc_id` to `data/sample-paper.doc_id` and skip re-indexing on reruns -- indexing is the most expensive one-time step per document.
- **Per-query cost**: the manual tree-search step (Section 5) makes 2 LLM calls per question (search + answer); the agentic step (Section 6) makes as many tool-calling round-trips as the agent decides it needs. Both are typically far cheaper than re-embedding a large corpus, but pricier per-query than a vector lookup once embeddings already exist.
- `.pageindex/` and `data/` are git-ignored -- they're local cache/output, not source.

## 9. Wrap-up

- We built a document tree locally with `PageIndexLocalClient`, inspected it, then retrieved answers two ways: a manual tree-search prompt, and a fully agentic multi-hop tool-calling flow -- no embeddings or vector store anywhere.
- For production use, PageIndex also offers a hosted **Cloud API** (`PageIndexCloudClient`) and an **MCP** server, so the same tool contracts and agent instructions work against a managed backend instead of local storage -- swap `PageIndexLocalClient()` for `PageIndexCloudClient(api_key=...)` and the rest of this notebook's code is unchanged.
- PageIndex also accepts Markdown input (`md_path`) if your source documents aren't PDFs.

**References**
- [VectifyAI/PageIndex](https://github.com/VectifyAI/PageIndex)
- [PageIndex docs](https://docs.pageindex.ai/)
- [Tree-search tutorial](https://github.com/VectifyAI/PageIndex/blob/main/examples/tutorials/tree-search/README.md)